# Notebook 2: Multi-Strategy Comparison (A/B/C Testing)
---
## Strategy
Run **3 different clustering approaches** on the same data and compare metrics.

### Option A: Tuned BERTopic (clean_text)
- Uses cleaned text without entity normalization
- Standard HDBSCAN clustering

### Option B: NER-Normalized BERTopic (ner_text)
- Uses entity-normalized text (Modi variants → pm_modi)
- Reduces cluster fragmentation

### Option C: Hybrid Bucketed BERTopic
- Pre-groups data into semantic buckets (politics, geopolitics, etc.)
- Runs BERTopic independently within each bucket
- Produces a hierarchical topic structure

## When to Use
- When you need to JUSTIFY which approach to use in production
- For stakeholder presentations comparing metrics
- Academic benchmarking

## Key Metrics Compared
- Silhouette Score, Noise Ratio, Topic Count, Dominance Ratio


In [ ]:
!pip install -q bertopic sentence-transformers hdbscan umap-learn pandas scikit-learn
print("Ready")

In [ ]:
import pandas as pd, numpy as np, re, html
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
import umap, hdbscan
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]

## Shared Setup: Load + Clean + Embed

In [ ]:
DOMAIN_STOPS = {"abp","shorts","viral","subscribe","youtube","news","live","at2","n18v"}
ENTITY_MAP = [(r"\bpm\s+modi\b|\bmodi\s+ji\b|\bnarendra\s+modi\b", " pm_modi "),
              (r"\brahul\s+gandhi\b", " rahul_gandhi "),
              (r"\bdonald\s+trump\b", " donald_trump ")]

def clean(text):
    if not isinstance(text, str): return ""
    text = html.unescape(text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"#(\w+)", r" \1 ", text)
    text = re.sub(r"[^\w\s\u0900-\u097F]", " ", text)
    tokens = [t for t in text.lower().split() if not (t.isascii() and len(t)<=2) and t not in DOMAIN_STOPS]
    return " ".join(tokens).strip()

def ner_clean(text):
    for pat, repl in ENTITY_MAP:
        text = re.sub(pat, repl, text, flags=re.I)
    return clean(text)

# --- UNIVERSAL DATA LOADER ---
import pandas as pd
import os

# Get the file extension
file_ext = os.path.splitext(filename)[1].lower()

if file_ext == '.xlsx':
    print("📂 Detected Excel file. Using read_excel...")
    df = pd.read_excel(filename)
elif file_ext == '.csv' or file_ext == '.txt':
    print("📂 Detected Text/CSV file. Using read_csv with encoding recovery...")
    try:
        df = pd.read_csv(filename, encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(filename, encoding='latin1')
        except:
            df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')
else:
    print(f"⚠️ Unknown format {file_ext}. Attempting generic read...")
    df = pd.read_csv(filename, sep="\v", engine="python", encoding='latin1')

# Ensure we have a title column
if "title" not in df.columns:
    if len(df.columns) == 1:
        df.columns = ["title"]
    else:
        df.columns = ["title"] + list(df.columns[1:])
# ------------------------------
if "title" not in df.columns: df.columns=["title"]+list(df.columns[1:])
df["clean_text"] = df["title"].fillna("").astype(str).map(clean)
df["ner_text"] = df["title"].fillna("").astype(str).map(ner_clean)
df = df[df["clean_text"]!=""].reset_index(drop=True)

embedder = SentenceTransformer("sentence-transformers/LaBSE")
clean_emb = embedder.encode(df["clean_text"].tolist(), show_progress_bar=True, batch_size=32)
ner_emb = embedder.encode(df["ner_text"].tolist(), show_progress_bar=True, batch_size=32)
print(f"Embedded {len(df)} docs")

## Option A: Tuned BERTopic (clean_text)

In [ ]:
model_a = BERTopic(embedding_model=embedder, min_topic_size=15, nr_topics="auto", verbose=True)
topics_a, _ = model_a.fit_transform(df["clean_text"].tolist(), clean_emb)
noise_a = round(100*(np.array(topics_a)==-1).mean(),1)
n_topics_a = len(set(topics_a))-1
print(f"Option A: {n_topics_a} topics, {noise_a}% noise")

## Option B: NER-Normalized BERTopic (ner_text)

In [ ]:
model_b = BERTopic(embedding_model=embedder, min_topic_size=15, nr_topics="auto", verbose=True)
topics_b, _ = model_b.fit_transform(df["ner_text"].tolist(), ner_emb)
noise_b = round(100*(np.array(topics_b)==-1).mean(),1)
n_topics_b = len(set(topics_b))-1
print(f"Option B: {n_topics_b} topics, {noise_b}% noise")

## Option C: Hybrid Bucketed BERTopic
Runs separate BERTopic per semantic bucket.

In [ ]:
BUCKETS = {
  "iran|israel|trump|war|military": "Geopolitiks",
  "election|bjp|congress|vote|candidate": "Elections",
  "modi|shah|gandhi|mamata|yogi": "Politicians",
}

def assign_bucket(text):
    for pattern, name in BUCKETS.items():
        if re.search(pattern, text, re.I): return name
    return "General"

df["bucket"] = df["ner_text"].map(assign_bucket)
all_topics_c = [-1]*len(df)
gid = 0
for bname, bdf in df.groupby("bucket"):
    if len(bdf)<15: continue
    idx = bdf.index.tolist()
    m = BERTopic(embedding_model=embedder, min_topic_size=10, verbose=False)
    t, _ = m.fit_transform(bdf["ner_text"].tolist(), ner_emb[idx])
    for i,ti in zip(idx,t):
        if ti!=-1: all_topics_c[i] = gid+ti
    gid += max(t)+1 if max(t)>=0 else 0

noise_c = round(100*sum(1 for t in all_topics_c if t==-1)/len(all_topics_c),1)
n_topics_c = len(set(all_topics_c))-1
print(f"Option C: {n_topics_c} topics, {noise_c}% noise")

## Comparison Table

In [ ]:
comparison = pd.DataFrame([
    {"Method": "A: Tuned BERTopic", "Topics": n_topics_a, "Noise %": noise_a},
    {"Method": "B: NER-Normalized", "Topics": n_topics_b, "Noise %": noise_b},
    {"Method": "C: Hybrid Bucketed", "Topics": n_topics_c, "Noise %": noise_c},
])
comparison

In [ ]:
## FINAL CLEAN STANDARDIZED EXPORT
# 1. Create a CLEAN Topic Label Map (No numbers, Space-separated, Title-case)
topic_info = model.get_topic_info()
label_map = {}
for _, row in topic_info.iterrows():
    t_id = row['Topic']
    raw_name = row['Name']
    # Remove the "0_" or "-1_" prefix and clean up underscores
    if "_" in raw_name:
        clean_name = raw_name.split("_", 1)[-1].replace("_", " ").title()
    else:
        clean_name = raw_name.title()
    label_map[t_id] = clean_name

df['topic_label'] = df['topic_id'].map(label_map)

# 2. Reorganize Columns
if 'ner_text' not in df.columns: df['ner_text'] = df['clean_text']
final_df = df.rename(columns={'title': 'raw_text'})

# 3. Selection (Exactly 5 columns)
cols_to_keep = ['raw_text', 'clean_text', 'ner_text', 'topic_id', 'topic_label']
final_df = final_df[cols_to_keep]

# 4. Save and Download
output_fn = "final_clustering_results.csv"
final_df.to_csv(output_fn, index=False)
print(f"Success! Exported {len(final_df)} titles with clean, unique labels.")
from google.colab import files
files.download(output_fn)